# 03 — Image Feature Extraction (Conjunctiva)
**AnemiaFusionNet — Stage 3**

Trains/fine-tunes a CNN backbone (EfficientNet-B0) on conjunctiva images to classify Anemic/Non-Anemic standalone on the GPU, and exports 256-d visual features for the multimodal fusion model.

## 3.1 Set Working Directory and Load Packages

In [1]:
import os
from pathlib import Path

# Change working directory to project root if executed from notebooks folder
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("Current working directory:", os.getcwd())

Current working directory: C:\Users\ratha\OneDrive\Desktop\datavidwan\New folder (2)\files


## 3.2 Prepare Data Loaders
We load the train, val, and test splits and instantiate `ConjunctivaDataset` with data augmentation for the training split.

In [2]:
import torch
import torchvision.transforms as T
from torch.utils.data import DataLoader
from src.datasets import ConjunctivaDataset

# Data Augmentation for training (mild only, color is clinical signal)
train_transform = T.Compose([
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.05, contrast=0.05),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

img_root = "data/raw/images"

train_dataset = ConjunctivaDataset("data/processed/train.csv", img_root, transform=train_transform)
val_dataset = ConjunctivaDataset("data/processed/val.csv", img_root, transform=val_transform)
test_dataset = ConjunctivaDataset("data/processed/test.csv", img_root, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

print(f"Batches in Train: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")

Batches in Train: 10 | Val: 3 | Test: 3


## 3.3 Initialize Model and Optimizer
We check for GPU (CUDA) availability and load the standalone `ImageClassifier` model.

In [3]:
import torch.nn as nn
from src.image_encoder import ImageClassifier

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training on device:", device)

model = ImageClassifier(embed_dim=256, num_classes=2, pretrained=True).to(device)

# Freeze the backbone features to prevent representation collapse/overfitting
for param in model.encoder.backbone.parameters():
    param.requires_grad = False

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3, weight_decay=1e-3)

Training on device: cuda


## 3.4 Model Training Standalone
We train the model on the GPU for 10 epochs, monitoring validation F1-score and saving the best checkpoint.

In [4]:
import time
from src.utils import calculate_metrics
from torch.optim.lr_scheduler import CosineAnnealingLR

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits, _ = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
    return running_loss / len(loader.dataset)

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    all_labels, all_preds, all_probs = [], [], []
    for images, labels in loader:
        images = images.to(device)
        logits, _ = model(images)
        probs = torch.softmax(logits, dim=-1)[:, 1]
        preds = logits.argmax(dim=-1)
        
        all_labels.extend(labels.numpy())
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
        
    return calculate_metrics(all_labels, all_preds, all_probs)

best_val_f1 = -1.0
epochs = 35
scheduler = CosineAnnealingLR(optimizer, T_max=epochs)

print("Starting training...")
for epoch in range(1, epochs + 1):
    start_time = time.time()
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
    val_metrics = evaluate(model, val_loader, device)
    scheduler.step()
    elapsed = time.time() - start_time
    
    print(f"Epoch {epoch:02d} | Loss: {train_loss:.4f} | Val F1: {val_metrics['f1']:.4f} | Acc: {val_metrics['accuracy']:.4f} | Time: {elapsed:.2f}s")
    
    if val_metrics["f1"] > best_val_f1 or (val_metrics["f1"] == 0.0 and best_val_f1 == -1.0):
        best_val_f1 = val_metrics["f1"]
        os.makedirs("models", exist_ok=True)
        torch.save(model.state_dict(), "models/image_classifier_best.pt")
        
print("Training complete. Best Validation F1:", best_val_f1)

Starting training...


Epoch 01 | Loss: 1.1908 | Val F1: 0.5957 | Acc: 0.4242 | Time: 1.30s


Epoch 02 | Loss: 0.7766 | Val F1: 0.0000 | Acc: 0.5758 | Time: 0.50s


Epoch 03 | Loss: 0.7304 | Val F1: 0.0000 | Acc: 0.5758 | Time: 0.49s


Epoch 04 | Loss: 0.6805 | Val F1: 0.0000 | Acc: 0.5758 | Time: 0.47s


...

Epoch 31 | Loss: 0.6892 | Val F1: 0.0000 | Acc: 0.5758 | Time: 0.49s


Epoch 32 | Loss: 0.6855 | Val F1: 0.0000 | Acc: 0.5758 | Time: 0.48s


Epoch 33 | Loss: 0.6863 | Val F1: 0.0000 | Acc: 0.5758 | Time: 0.46s


Epoch 34 | Loss: 0.6829 | Val F1: 0.0000 | Acc: 0.5758 | Time: 0.46s


Epoch 35 | Loss: 0.6788 | Val F1: 0.0000 | Acc: 0.5758 | Time: 0.47s
Training complete. Best Validation F1: 0.5957446808510638


## 3.5 Load Best Model and Export Embeddings
We load the best classifier state dictionary and export the visual embeddings for all patient images in the `master_dataset.csv` to `data/embeddings/image_embeddings.pt`.

In [5]:
# Load best model weights
model.load_state_dict(torch.load("models/image_classifier_best.pt"))
model.eval()

# Load entire master dataset sequentially
master_dataset = ConjunctivaDataset("data/processed/master_dataset.csv", img_root, transform=val_transform)
master_loader = DataLoader(master_dataset, batch_size=16, shuffle=False)

embeddings = []
labels_list = []

print("Exporting image embeddings...")
with torch.no_grad():
    for images, labels in master_loader:
        images = images.to(device)
        _, emb = model(images)
        embeddings.append(emb.cpu())
        labels_list.append(labels)
        
all_embeddings = torch.cat(embeddings, dim=0)
all_labels = torch.cat(labels_list, dim=0)

print("Exported embeddings shape:", all_embeddings.shape)  # Should be (N, 256)
print("Exported labels shape:", all_labels.shape)

os.makedirs("data/embeddings", exist_ok=True)
torch.save(all_embeddings, "data/embeddings/image_embeddings.pt")
torch.save(all_labels, "data/embeddings/image_labels.pt")
print("Saved image embeddings and labels to data/embeddings/")

Exporting image embeddings...


Exported embeddings shape: torch.Size([217, 256])
Exported labels shape: torch.Size([217])
Saved image embeddings and labels to data/embeddings/
